# A History of Solar Activity over Millennia — Implementation / 구현

**Paper**: Usoskin, I. G. (2023). *A History of Solar Activity over Millennia*. Living Reviews in Solar Physics, 20:2. DOI: 10.1007/s41116-023-00036-z

**Update of Paper #52** (Usoskin 2017).

This notebook implements the key algorithms and concepts from Usoskin (2023) covering:
1. Synthetic 10-kyr cosmogenic isotope ($^{10}$Be, $^{14}$C) time series with embedded Grand Minima/Maxima and Miyake events
2. FFT / wavelet analysis of Schwabe, Gleissberg, de Vries/Suess, Hallstatt periodicities
3. ISN v.2 vs v.1 reconciliation (Clette–Lefèvre 2016)
4. Miyake event cosmogenic production via CRBM (Cosmic-Ray Box Model)
5. Grand Minima waiting-time distribution
6. Reservoir box-model for $^{14}$C (Oeschger–Siegenthaler style)

이 노트북은 Usoskin (2023)의 핵심 알고리즘과 개념을 구현합니다.

- 합성 10 kyr 우주 생성 동위원소($^{10}$Be, $^{14}$C) 시계열 (그랜드 미니마/맥시마 + Miyake 사건 내장)
- Schwabe, Gleissberg, de Vries/Suess, Hallstatt 주기성의 FFT/웨이블릿 분석
- ISN v.2 대 v.1 재조정 (Clette–Lefèvre 2016)
- CRBM (Cosmic-Ray Box Model)로 계산되는 Miyake 사건 생성률
- 그랜드 미니마 대기 시간 분포
- $^{14}$C용 저장소 상자 모델 (Oeschger–Siegenthaler 스타일)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.stats import expon, weibull_min

plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 11
rng = np.random.default_rng(42)

## Part 1: Synthetic 10-kyr Cosmogenic Isotope Time Series / 합성 10 kyr 우주 생성 동위원소 시계열

We construct a 10,000-year annual time series of cosmogenic-isotope production rate $Q(t)$ that embeds:
- **11-yr Schwabe** oscillation
- **~90-yr Gleissberg** modulation
- **~210-yr de Vries/Suess** cycle
- **~2400-yr Hallstatt** modulation of grand-minima occurrence
- **Grand Minima** (25 over 11 kyr ≈ 1 per 400 yr with clustering)
- **Grand Maxima** (23 over 11 kyr)
- **Miyake events** (5 confirmed: 774 CE, 993 CE, 660 BCE, 5259 BCE, 7176 BCE)

11년 Schwabe 진동, ~90년 Gleissberg 변조, ~210년 de Vries/Suess 주기, ~2400년 Hallstatt 그랜드 미니마 변조, 그랜드 미니마 25회, 그랜드 맥시마 23회, 5개 확인된 Miyake 사건을 포함.

In [ ]:
def synthetic_cosmogenic_series(n_years: int = 10000, dt: float = 1.0):
    """Generate synthetic cosmogenic isotope production rate time series.

    Implements a phenomenological model following Usoskin (2023) with
    Schwabe, Gleissberg, de Vries/Suess, and Hallstatt periodicities
    plus embedded Grand Minima/Maxima and Miyake events.

    Args:
        n_years: Length of the series in years.
        dt: Time step in years.

    Returns:
        time: Array of time values (years).
        q: Production rate (atoms/cm^2/s) normalized around 1.75.
        phi: Modulation potential (MV).
        miyake_events: List of (time, amplitude) tuples for spike events.
    """
    time = np.arange(0, n_years, dt)
    # Base modulation potential ~ 500 MV quiet, 1200 MV max, ~800 mean
    phi = 700.0 + np.zeros_like(time)
    # Schwabe 11-yr
    phi += 250.0 * np.sin(2 * np.pi * time / 11.0)
    # Gleissberg ~90-yr modulation on Schwabe envelope
    phi += 80.0 * np.sin(2 * np.pi * time / 90.0)
    # de Vries/Suess ~210-yr
    phi += 60.0 * np.sin(2 * np.pi * time / 210.0)
    # Hallstatt ~2400-yr
    hallstatt = 100.0 * np.sin(2 * np.pi * time / 2400.0)
    phi += hallstatt
    # Add red noise
    noise = rng.normal(0, 40, size=time.shape)
    phi += np.convolve(noise, np.ones(5) / 5, mode='same')

    # Embed 25 Grand Minima clustered near Hallstatt lows
    n_minima = 25
    gm_centers = []
    for _ in range(n_minima):
        # Bias toward Hallstatt lows (negative hallstatt term)
        candidate = rng.integers(0, n_years)
        for _ in range(10):
            alt = rng.integers(0, n_years)
            if hallstatt[int(alt)] < hallstatt[int(candidate)]:
                candidate = alt
        gm_centers.append(int(candidate))
    for c in gm_centers:
        duration = rng.integers(40, 120)
        lo = max(0, c - duration // 2)
        hi = min(n_years, c + duration // 2)
        phi[lo:hi] -= 500.0  # deep minimum

    # Embed 23 Grand Maxima
    for _ in range(23):
        c = rng.integers(0, n_years)
        duration = rng.integers(30, 80)
        lo = max(0, c - duration // 2)
        hi = min(n_years, c + duration // 2)
        phi[lo:hi] += 350.0

    phi = np.clip(phi, 100, 2500)

    # Convert modulation potential to production rate Q (phenomenological)
    # Usoskin (2023): Q decreases with increasing phi; for 14C, Q ~ 1.5-2.5 atoms/cm2/s
    q = 2.5 - 1.2 * (phi - 300) / 1000.0
    q = np.clip(q, 0.8, 2.8)

    # Miyake events (year index within 10 kyr; realistic ~5 events)
    miyake_events = [
        (9175, 70.0),   # 7176 BCE (if 0 = 8176 BCE approx)
        (7092, 65.0),   # 5259 BCE
        (4010, 52.0),   # 660 BCE
        (5444, 70.0),   # 774/5 CE
        (5663, 45.0),   # 993 CE
    ]
    for yr, amp in miyake_events:
        if 0 <= yr < n_years:
            # Sharp spike with exponential decay (carbon cycle response)
            decay = np.exp(-np.arange(0, 30) / 5.0)
            end = min(yr + 30, n_years)
            q[yr:end] += amp * decay[: end - yr] / 10.0

    return time, q, phi, miyake_events

In [ ]:
time, q, phi, miyake_events = synthetic_cosmogenic_series(n_years=10000)

fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
axes[0].plot(time, phi, 'b-', lw=0.5)
axes[0].set_ylabel('Modulation potential $\\phi$ (MV)')
axes[0].set_title('Synthetic 10-kyr solar-activity reconstruction / 합성 10 kyr 태양 활동 복원')
axes[0].axhline(500, color='gray', ls='--', alpha=0.5, label='quiet')
axes[0].axhline(1200, color='red', ls='--', alpha=0.5, label='max')
axes[0].legend()

axes[1].plot(time, q, 'g-', lw=0.3)
axes[1].set_ylabel('$Q(^{14}$C) (atoms/cm$^2$/s)')
for yr, amp in miyake_events:
    axes[1].axvline(yr, color='red', alpha=0.5, lw=1)
    axes[1].text(yr, q.max() * 0.9, f'Miyake', rotation=90, fontsize=8, color='red')

# Running 50-year smoothed SN-equivalent
window = 50
sn_proxy = np.convolve(80 - 0.1 * phi, np.ones(window) / window, mode='same')
sn_proxy = np.clip(sn_proxy, 0, None)
axes[2].plot(time, sn_proxy, 'k-', lw=0.6)
axes[2].axhline(50, color='red', ls=':', alpha=0.6, label='Grand Max threshold (SN > 50)')
axes[2].axhline(15, color='blue', ls=':', alpha=0.6, label='Grand Min threshold (SN < 15)')
axes[2].set_ylabel('Smoothed SN proxy')
axes[2].set_xlabel('Time (years from start)')
axes[2].legend()
plt.tight_layout()
plt.show()

## Part 2: FFT / Wavelet Analysis — Hallstatt, Gleissberg, Schwabe / FFT·웨이블릿 분석

Reproduce Fig. 21–22 of Usoskin (2023): detect the Schwabe, Gleissberg, de Vries/Suess, and Hallstatt peaks in the wavelet power spectrum.

Usoskin (2023)의 Fig. 21–22 재현: 웨이블릿 파워 스펙트럼에서 Schwabe, Gleissberg, de Vries/Suess, Hallstatt 피크 탐지.

In [ ]:
def morlet_wavelet_power(x: np.ndarray, dt: float = 1.0, periods: np.ndarray = None):
    """Compute Morlet wavelet power spectrum.

    Args:
        x: Input time series.
        dt: Sample spacing.
        periods: Array of periods to evaluate.

    Returns:
        power: 2D array (len(periods), len(x)) of wavelet power.
        periods: Same as input for convenience.
    """
    if periods is None:
        periods = np.logspace(np.log10(4), np.log10(4000), 60)
    omega0 = 6.0
    widths = omega0 * periods / (2 * np.pi * dt)
    # scipy.signal.cwt with Morlet equivalent
    coef = signal.cwt(x - x.mean(), signal.morlet2, widths, w=omega0)
    power = np.abs(coef) ** 2
    return power, periods

In [ ]:
# Use q with Schwabe signature subtracted out for visibility
x_for_wavelet = q - np.mean(q)
power, periods = morlet_wavelet_power(x_for_wavelet, dt=1.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Wavelet power heatmap
im = ax1.pcolormesh(time, periods, np.log10(power + 1e-12), shading='auto', cmap='viridis')
ax1.set_yscale('log')
ax1.set_xlabel('Time (yr)')
ax1.set_ylabel('Period (yr)')
ax1.set_title('Morlet wavelet power spectrum / Morlet 웨이블릿 파워 스펙트럼')
for p_mark, name in [(11, 'Schwabe'), (90, 'Gleissberg'), (210, 'de Vries'), (2400, 'Hallstatt')]:
    ax1.axhline(p_mark, color='white', ls='--', alpha=0.7)
    ax1.text(time[-1] * 0.02, p_mark, name, fontsize=9, color='white')
plt.colorbar(im, ax=ax1, label='log10 power')

# Global (time-averaged) wavelet spectrum
global_power = power.mean(axis=1)
ax2.plot(periods, global_power, 'k-', lw=1.5)
ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel('Period (yr)')
ax2.set_ylabel('Global wavelet power')
ax2.set_title('Global wavelet spectrum / 전역 웨이블릿 스펙트럼')
for p_mark, name in [(11, 'Schwabe'), (90, 'Gleissberg'), (210, 'de Vries'), (2400, 'Hallstatt')]:
    ax2.axvline(p_mark, color='red', ls='--', alpha=0.5)
    ax2.text(p_mark, global_power.max() * 1.2, name, rotation=45, fontsize=9, color='red')
ax2.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## Part 3: ISN v.2 vs v.1 — Clette–Lefèvre (2016) Revision / ISN v.2 대 v.1 — Clette–Lefèvre (2016) 개정

The ISN v.2 rescaling: $R_z^{(v.2)} \approx 1.667 \times R_z^{(v.1)}$ (except for Waldmeier discontinuity correction ~1947). This is a major update since Paper #52.

ISN v.2 재조정: $R_z^{(v.2)} \approx 1.667 \times R_z^{(v.1)}$ (단 1947년 Waldmeier 불연속 보정 제외). Paper #52 대비 주요 갱신.

In [ ]:
def isn_v1_to_v2(rz_v1: np.ndarray, years: np.ndarray) -> np.ndarray:
    """Convert ISN v.1 to ISN v.2 following Clette–Lefèvre (2016).

    Applies the constant factor 1.667 (=1/0.6) rescaling to Wolfer,
    plus a Waldmeier discontinuity correction post-1947 (~15% reduction).

    Args:
        rz_v1: ISN v.1 values.
        years: Year for each sample.

    Returns:
        ISN v.2 values.
    """
    rz_v2 = 1.667 * rz_v1
    # Waldmeier discontinuity — Clette et al. (2014): ~15% overestimate 1947+
    post_1947 = years >= 1947
    rz_v2[post_1947] *= 0.85
    return rz_v2


# Synthetic example using 1750-2020 period with realistic cycle pattern
years_full = np.arange(1750, 2024)
phase = 2 * np.pi * (years_full - 1755) / 11.0
rz_v1 = (60 + 40 * np.sin(phase)) * np.clip(np.sin(np.pi * (years_full - 1750) / 200) + 0.7, 0.1, None)
rz_v1 += 20 * np.sin(2 * np.pi * (years_full - 1750) / 90.0)
rz_v1 = np.clip(rz_v1, 0, None)
# Emulate Maunder Min effect at start (already part of smooth envelope)
rz_v2 = isn_v1_to_v2(rz_v1.copy(), years_full)

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(years_full, rz_v1 * 1.67, 'k-', lw=0.8, label='ISN v.1 $\\times$ 1.67')
ax.plot(years_full, rz_v2, 'r-', lw=0.8, label='ISN v.2 (with Waldmeier correction)')
ax.axvline(1947, color='gray', ls=':', alpha=0.7, label='Waldmeier discontinuity')
ax.set_xlabel('Year')
ax.set_ylabel('Sunspot number')
ax.set_title('ISN v.1 vs v.2 (Clette–Lefèvre 2016) / ISN v.1 대 v.2')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Pre-1947 ratio v.2/(v.1*1.667): {np.mean(rz_v2[years_full < 1947] / (rz_v1[years_full < 1947] * 1.667)):.3f}")
print(f"Post-1947 ratio v.2/(v.1*1.667): {np.mean(rz_v2[years_full >= 1947] / (rz_v1[years_full >= 1947] * 1.667)):.3f}")

## Part 4: Miyake Event CRBM Production / Miyake 사건 CRBM 생성률

For the 774/5 CE event, the cosmic-ray box model with an ESPE integral fluence $F(>200\text{ MeV}) \approx 10^{10}$ cm$^{-2}$ yields the observed $^{14}$C enhancement factor 3.9 × background.

$$\Delta Q_{14\mathrm{C}}^{\mathrm{ESPE}} = \int_{E_{\min}}^{\infty} F(>E)\,Y_{14\mathrm{C}}(E)\,dE \cdot \delta(t - t_{\mathrm{ESPE}})$$

Then the carbon cycle response produces a delayed, decaying $\Delta^{14}$C spike.

774/5 CE 사건의 경우, 우주선 상자 모델(CRBM)에서 ESPE 적분 플루언스 $F(>200\text{ MeV}) \approx 10^{10}$ cm$^{-2}$로 관측된 $^{14}$C 배경 대비 3.9배 증가를 재현할 수 있다.

In [ ]:
def espe_cosmogenic_production(fluence_200mev: float = 1e10,
                                effective_energy_gev: float = 0.234,
                                yield_14c_per_proton: float = 2.3e-4) -> dict:
    """Compute ESPE-driven cosmogenic isotope production enhancement.

    Args:
        fluence_200mev: Event-integrated proton fluence > 200 MeV (cm^-2).
        effective_energy_gev: SEP effective production energy for 14C.
        yield_14c_per_proton: Integrated yield atoms 14C per primary proton.

    Returns:
        Dict with total extra 14C, 10Be, 36Cl atoms/cm^2 above background.
    """
    # Background annual global production rate Q0
    q0_14c = 1.75  # atoms/cm2/s
    q0_10be = 0.018  # atoms/cm2/s
    q0_36cl = 1.9e-3  # atoms/cm2/s
    seconds_per_year = 3.156e7
    annual_q0_14c = q0_14c * seconds_per_year  # atoms/cm2/yr
    annual_q0_10be = q0_10be * seconds_per_year
    annual_q0_36cl = q0_36cl * seconds_per_year

    # ESPE extra production (simplified: fluence x yield)
    extra_14c = fluence_200mev * yield_14c_per_proton
    extra_10be = fluence_200mev * 2.0e-5  # roughly consistent with Mekhaldi 2015 ratios
    extra_36cl = fluence_200mev * 3.0e-6

    return {
        'extra_14c_cm2': extra_14c,
        'extra_10be_cm2': extra_10be,
        'extra_36cl_cm2': extra_36cl,
        'enhance_14c': 1 + extra_14c / annual_q0_14c,
        'enhance_10be': 1 + extra_10be / annual_q0_10be,
        'enhance_36cl': 1 + extra_36cl / annual_q0_36cl,
    }


result = espe_cosmogenic_production(fluence_200mev=1e10)
print("=== 774/5 CE Miyake Event CRBM Production ===")
for k, v in result.items():
    print(f"  {k}: {v:.3g}")

# Compare to reported Mekhaldi et al. 2015 values
print("\n=== Observed (Mekhaldi et al. 2015) ===")
print("  14C enhancement factor: 3.9")
print("  10Be enhancement factor: 3.4")
print("  36Cl enhancement factor: 6.3")

In [ ]:
# Plot the Delta14C time response to a 774 CE-like injection
def carbon_cycle_response_to_spike(t_event: int, amp: float, n_years: int = 100) -> np.ndarray:
    """Simple carbon-cycle decay response for delta14C.

    Assumes instantaneous stratospheric injection followed by mixing to
    troposphere and ocean. Response modeled as fast rise (1 yr) + two-exponential
    decay (biosphere: tau=20 yr; ocean mixing: tau=500 yr).

    Args:
        t_event: Year of injection (array index).
        amp: Peak Delta14C enhancement in permil.
        n_years: Total simulation length.

    Returns:
        Array of delta14C enhancement above background.
    """
    t = np.arange(n_years)
    d14c = np.zeros(n_years)
    tau_fast = 20.0
    tau_slow = 500.0
    dt_from_event = t - t_event
    mask = dt_from_event >= 0
    d14c[mask] = amp * (0.7 * np.exp(-dt_from_event[mask] / tau_fast) +
                        0.3 * np.exp(-dt_from_event[mask] / tau_slow))
    return d14c


t_plot = np.arange(770, 800)
d14c_response = carbon_cycle_response_to_spike(t_event=5, amp=15.0, n_years=len(t_plot))

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(t_plot, d14c_response, 'r-', lw=2, label='CRBM response (model)')
ax.axvline(775, color='gray', ls='--', label='774/5 CE injection')
ax.set_xlabel('Year (CE)')
ax.set_ylabel('$\\Delta^{14}$C enhancement (‰)')
ax.set_title('774/5 CE Miyake event $\\Delta^{14}$C response / Miyake 사건 $\\Delta^{14}$C 응답')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Part 5: Grand Minima Waiting-Time Distribution / 그랜드 미니마 대기 시간 분포

Reproduce Usoskin et al. (2007) analysis: compare observed waiting-time distribution (from Table 3: 25 grand minima over 11 kyr) vs memoryless Poisson null hypothesis. Expected deviation: sub-exponential (clustering + long gaps).

Usoskin et al. (2007) 분석 재현: 관측된 대기 시간 분포 (Table 3의 11 kyr 내 25개 그랜드 미니마)와 무기억 Poisson 귀무가설 비교. 예상 편차: sub-exponential (군집화 + 긴 무사건 구간).

In [ ]:
# Grand Minima centers from Table 3 of Usoskin (2023) — dates in -BC/AD (kyr):
grand_minima_centers_bc_ad = np.array([
    1680, 1470, 1310, 1030, 690, -360, -750, -1385,
    -2450, -2855, -3325, -3495, -3620, -4220, -4315,
    -5195, -5300, -5460, -5610, -6385, -7035, -7305,
    -7515, -8215, -9165
])
grand_minima_centers_bc_ad = np.sort(grand_minima_centers_bc_ad)
waiting_times = np.diff(grand_minima_centers_bc_ad)

print(f"Number of grand minima: {len(grand_minima_centers_bc_ad)}")
print(f"Number of waiting intervals: {len(waiting_times)}")
print(f"Mean waiting time: {waiting_times.mean():.0f} yr")
print(f"Median waiting time: {np.median(waiting_times):.0f} yr")
print(f"Min / Max: {waiting_times.min()} / {waiting_times.max()} yr")

# Fit exponential (Poisson null) and Weibull
lambda_inv = 1.0 / waiting_times.mean()
t_plot = np.linspace(0, waiting_times.max() * 1.2, 200)
exp_pdf = lambda_inv * np.exp(-lambda_inv * t_plot)

shape, loc, scale = weibull_min.fit(waiting_times, floc=0)
weibull_pdf = weibull_min.pdf(t_plot, shape, loc=loc, scale=scale)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.hist(waiting_times, bins=10, density=True, color='steelblue', alpha=0.7, edgecolor='black')
ax1.plot(t_plot, exp_pdf, 'r-', lw=2, label=f'Exponential (mean {waiting_times.mean():.0f})')
ax1.plot(t_plot, weibull_pdf, 'g--', lw=2, label=f'Weibull (shape {shape:.2f})')
ax1.set_xlabel('Waiting time $\\Delta t$ (yr)')
ax1.set_ylabel('PDF')
ax1.set_title('Grand Minima waiting-time distribution / 그랜드 미니마 대기 시간')
ax1.legend()

# Survival function (log scale) — key diagnostic
sorted_wt = np.sort(waiting_times)
sf_obs = 1 - np.arange(1, len(sorted_wt) + 1) / len(sorted_wt)
ax2.semilogy(sorted_wt, sf_obs, 'ko-', label='Observed', markersize=6)
ax2.semilogy(t_plot, np.exp(-lambda_inv * t_plot), 'r-', lw=2, label='Poisson exp')
ax2.set_xlabel('$\\Delta t$ (yr)')
ax2.set_ylabel('Survival function $P(\\Delta t > T)$')
ax2.set_title('Survival function: deviation from Poisson / 생존 함수: Poisson 편차')
ax2.legend()
ax2.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

print(f"\nWeibull shape parameter: {shape:.3f} (< 1 means clustering)")
print(f"Weibull scale parameter: {scale:.1f} yr")

## Part 6: Reservoir Box Model for $^{14}$C / $^{14}$C 저장소 상자 모델

Implement a simplified 4-box carbon-cycle model (atmosphere–biosphere–ocean mixed layer–deep ocean) following Oeschger–Siegenthaler (1980). This is the mathematical basis of the Bern3D-LPJ model used in Usoskin (2023).

$$\frac{dN_i}{dt} = Q_i + \sum_j k_{ji} N_j - N_i \sum_j k_{ij} - \lambda N_i$$

where $N_i$ is the $^{14}$C inventory in reservoir $i$, $k_{ij}$ is the exchange rate, and $\lambda = \ln 2 / 5730$ yr$^{-1}$ is the decay constant.

Oeschger–Siegenthaler (1980)을 따라 간략화된 4-상자 탄소 순환 모델 (대기–생물권–해양 표층–해양 심층)을 구현. 이는 Usoskin (2023)이 사용한 Bern3D-LPJ 모델의 수학적 기반.

In [ ]:
def simulate_carbon_cycle(production_rate: np.ndarray, dt: float = 1.0) -> np.ndarray:
    """Simulate Delta14C atmospheric response to time-varying production rate.

    4-box model (atm, biosphere, ocean surface, deep ocean). Parameters based on
    Siegenthaler et al. 1980 with simplifications.

    Args:
        production_rate: Time series of 14C production (atoms/cm^2/s).
        dt: Time step (years).

    Returns:
        delta14c_atm: Array of delta14C in atmosphere (permil).
    """
    n = len(production_rate)
    # Inventories (atoms) — initialized at quasi-steady state
    N = np.array([0.62, 1.86, 0.93, 37.50])  # atm, bio, oce_surf, oce_deep (rel units)
    N0 = N.copy()

    # Exchange rate coefficients k_ij (yr^-1) — fluxes normalized
    k_atm_bio = 1.0 / 40.0
    k_bio_atm = 1.0 / 50.0
    k_atm_oce = 1.0 / 8.0
    k_oce_atm = 1.0 / 9.0
    k_ocesurf_ocedeep = 1.0 / 300.0
    k_ocedeep_ocesurf = 1.0 / 1500.0

    decay = np.log(2) / 5730.0

    delta14c_atm = np.zeros(n)

    for t in range(n):
        # Production into atmosphere
        prod_atm = production_rate[t]
        # Exchanges
        flux_atm = prod_atm - N[0] * (k_atm_bio + k_atm_oce) + N[1] * k_bio_atm + N[2] * k_oce_atm - decay * N[0]
        flux_bio = N[0] * k_atm_bio - N[1] * k_bio_atm - decay * N[1]
        flux_oces = N[0] * k_atm_oce - N[2] * (k_oce_atm + k_ocesurf_ocedeep) + N[3] * k_ocedeep_ocesurf - decay * N[2]
        flux_oced = N[2] * k_ocesurf_ocedeep - N[3] * k_ocedeep_ocesurf - decay * N[3]
        N[0] += dt * flux_atm
        N[1] += dt * flux_bio
        N[2] += dt * flux_oces
        N[3] += dt * flux_oced
        # Relative Delta14C (scaled)
        delta14c_atm[t] = 1000.0 * (N[0] / N0[0] - 1.0)

    return delta14c_atm

In [ ]:
# Simulate 500-year response to a Miyake-like spike on top of constant production
n_yr = 500
prod = np.ones(n_yr) * 1.75  # atoms/cm2/s
prod[100] += 20.0  # Miyake spike — 20x annual production in one year

d14c = simulate_carbon_cycle(prod)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
ax1.plot(np.arange(n_yr), prod, 'r-', lw=1)
ax1.set_ylabel('$^{14}$C production rate (atoms/cm$^2$/s)')
ax1.set_title('Atmospheric $^{14}$C response to Miyake-like injection / Miyake 주입에 대한 대기 $^{14}$C 응답')
ax1.axvline(100, color='gray', ls='--', label='Injection')
ax1.legend()

ax2.plot(np.arange(n_yr), d14c, 'b-', lw=1)
ax2.set_xlabel('Time (yr)')
ax2.set_ylabel('$\\Delta^{14}$C atmosphere (‰)')
ax2.axvline(100, color='gray', ls='--')
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Peak Delta14C after injection: {d14c[101:110].max():.2f} per mille")
print(f"Decay time to half peak: ~{np.argmax(d14c[100:] < d14c[100:].max() / 2)} yr")

## Summary / 요약

| Concept / 개념 | This Paper (Usoskin 2023) / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Sunspot series / 흑점 시리즈 | ISN v.2 + GSN (U16 preferred) | SILSO (https://sidc.be/silso) |
| CR modulation / CR 변조 | Force-field $\phi$ parameter | Drift model (Potgieter 2013), Parker eq. |
| ¹⁴C production / ¹⁴C 생성 | CRAC:14C (Poluianov 2016) | GEANT4-based full Monte Carlo |
| Carbon cycle / 탄소 순환 | Bern3D-LPJ (Roth & Joos 2013) | CMIP6 ESMs (earth system models) |
| Grand Minima census / 그랜드 미니마 목록 | 25 over 11 kyr (Table 3) | IntCal20 + new U16 reconstruction |
| Miyake events / Miyake 사건 | 5 confirmed + 3 candidates | NEEM/NGRIP/WDC ice cores cross-check |
| SEP worst case / SEP 최악 시나리오 | 774/5 CE ~70× GLE #5 | Used for spaceflight dose limits (NASA) |
| Solar cycle prediction / 태양 주기 예측 | Cycle 25: moderate | Jiang et al. 2023, Nandy 2021 |

### Paper #52 → #81 Key Changes / Paper #52 대비 주요 변경사항

1. **ISN v.2 + GSN revisions** (§2.2.1) — 40-page update due to Clette et al. 2014, 2016; sunspot community converging on consensus
2. **Miyake event catalog** — expanded from 2 confirmed (774, 993 CE) to 5 confirmed + 3 candidates; 660 BCE, 5259 BCE, 7176 BCE newly added
3. **Nitrate proxy dismissed** (§5.4) — Duderstadt et al. 2016 + Mekhaldi et al. 2017 show polar ice nitrate is NOT an SEP proxy — reversal of earlier practice
4. **Cycle 25 context** — confirmation that Modern Grand Maximum has ended
5. **Carbon cycle** — Bern3D-LPJ (Roth & Joos 2013) now standard replacing simpler box models
6. **References**: 474 → 534

1. **ISN v.2 + GSN 개정** (§2.2.1) — Clette et al. 2014, 2016에 따른 40쪽 갱신; 흑점 공동체 합의 수렴 중
2. **Miyake 사건 카탈로그** — 2개 확인(774, 993 CE)에서 5개 확인 + 3개 후보로 확장; 660 BCE, 5259 BCE, 7176 BCE 신규
3. **Nitrate 프록시 폐기** (§5.4) — Duderstadt et al. 2016과 Mekhaldi et al. 2017이 극관 빙하 질산염이 SEP 프록시가 아님을 입증
4. **25주기 맥락** — 현대 그랜드 맥시마 종료 확인
5. **탄소 순환** — Bern3D-LPJ (Roth & Joos 2013)이 간단한 상자 모델 대체
6. **참고문헌**: 474 → 534